# Multi-Agent Personal Stylist & Inventory Manager

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/robin-mongodb/mongochain/blob/main/examples/verticals/retail/multi-agent_demo.ipynb)

A multi-agent system demonstrating a feedback loop between style suggestions and inventory constraints.

## Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                           User Request                              │
│   "I need an outfit for a summer wedding, delivered by Friday"      │
└─────────────────────────────────┬───────────────────────────────────┘
                                  │
                                  ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    Agent A: The Stylist                             │
│  • User-facing, creative persona                                    │
│  • Suggests items based on occasion, trends, and style              │
│  • Tool 1: search_and_check_availability (quick product discovery)  │
│  • Tool 2: check_with_stock_keeper (agent-to-agent handoff)         │
└─────────────────────────────────┬───────────────────────────────────┘
                                  │ check_with_stock_keeper
                                  │ (calls stock_keeper.chat())
                                  ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    Agent B: The Stock Keeper                        │
│  • Backend inventory specialist                                     │
│  • ONE combined tool: check_inventory_and_shipping                  │
│  • (Combined because agents can only call 1 tool per response)      │
│  • Builds memories from agent-to-agent queries                      │
└─────────────────────────────────┬───────────────────────────────────┘
                                  │
                                  ▼
┌─────────────────────────────────────────────────────────────────────┐
│                         Feedback Loop                               │
│  If item unavailable → Stylist finds visual alternative             │
│  If shipping too slow → Stylist suggests expedited or swap          │
│  If all good → Complete the look and confirm order details          │
└─────────────────────────────────────────────────────────────────────┘
```

## Features Demonstrated
- **Multi-Agent Collaboration**: Stylist and Stock Keeper working together via tool-based handoff
- **Agent-to-Agent Communication**: `check_with_stock_keeper` calls `stock_keeper.chat()` — both agents build memories
- **Feedback Loop**: Style suggestions validated against real inventory constraints
- **Memory Sharing via `collaborators`**: Agents can read each other's memories about users

## Prerequisites
- MongoDB Atlas cluster
- Voyage AI API key
- OpenAI API key


## 1. Setup


In [ ]:
# Install mongochain from GitHub
%pip install -q git+https://github.com/robin-mongodb/mongochain.git


In [ ]:
from getpass import getpass

# API Keys (input hidden)
MONGO_URI = getpass("MongoDB URI: ")
VOYAGE_API_KEY = getpass("Voyage AI Key: ")
OPENAI_API_KEY = getpass("OpenAI API Key: ")


## 2. Seed Fashion Product Catalog

Create a fashion catalog with:
- SKU identifiers for inventory tracking
- Style metadata (occasion, season, trend status)
- Visual descriptions for semantic search
- Available sizes and colors


### 2a. Initialize Database Connections

Connect to MongoDB Atlas and set up collections for the product catalog and inventory system.

In [ ]:
from pymongo import MongoClient
import voyageai
from datetime import datetime

# Connect to MongoDB
client = MongoClient(MONGO_URI)

# Stylist's database - product catalog
stylist_db = client["personal_stylist"]
products = stylist_db["fashion_catalog"]

# Stock Keeper's database - inventory
inventory_db = client["stock_keeper"]
inventory = inventory_db["inventory"]
warehouses = inventory_db["warehouses"]

# Initialize Voyage for embeddings
voyage = voyageai.Client(api_key=VOYAGE_API_KEY)

# Clear existing data
products.delete_many({})
inventory.delete_many({})
warehouses.delete_many({})

print("✅ Database connections established")



### Fashion product catalog - wedding-appropriate items. 
Add your own data if needed

In [ ]:
# Fashion product catalog - wedding-appropriate items
FASHION_CATALOG = [
    # Men's Suits & Blazers
    {
        "sku": "MEN-SUIT-001",
        "name": "Coastal Linen Suit",
        "category": "Men's Suits",
        "price": 289.00,
        "description": "Relaxed-fit linen suit in a breezy sand color. Unlined blazer with natural texture, perfect for beach and garden weddings. Lightweight and breathable for summer events. Features a modern slim lapel and double-vent back.",
        "colors": ["Sand", "Light Blue", "Sage Green"],
        "sizes": ["S", "M", "L", "XL"],
        "occasions": ["wedding", "garden party", "summer formal"],
        "season": "summer",
        "trending": True,
        "style_tags": ["relaxed", "natural", "coastal", "breathable"]
    },
    {
        "sku": "MEN-SUIT-002",
        "name": "Floral Jacquard Blazer",
        "category": "Men's Blazers",
        "price": 195.00,
        "description": "Statement blazer with subtle floral jacquard pattern. Navy base with tonal botanical motifs. Perfect for making an entrance at summer celebrations. Pair with neutral trousers for balance.",
        "colors": ["Navy Floral", "Burgundy Floral"],
        "sizes": ["S", "M", "L", "XL"],
        "occasions": ["wedding", "cocktail party", "summer formal"],
        "season": "summer",
        "trending": True,
        "style_tags": ["statement", "bold", "floral", "fashion-forward"]
    },
    {
        "sku": "MEN-SUIT-003",
        "name": "Classic Seersucker Suit",
        "category": "Men's Suits",
        "price": 245.00,
        "description": "Traditional blue and white seersucker suit with a Southern charm. Puckered cotton fabric naturally stays cool. Two-button closure, partially lined. A timeless choice for summer weddings.",
        "colors": ["Blue/White Stripe", "Grey/White Stripe"],
        "sizes": ["S", "M", "L", "XL", "XXL"],
        "occasions": ["wedding", "Kentucky Derby", "summer formal"],
        "season": "summer",
        "trending": False,
        "style_tags": ["classic", "southern", "preppy", "traditional"]
    },
    {
        "sku": "MEN-BLAZER-001",
        "name": "Knit Cotton Blazer",
        "category": "Men's Blazers",
        "price": 165.00,
        "description": "Unstructured knit blazer in breathable cotton. Feels like a cardigan, looks like a blazer. Perfect for outdoor summer events where comfort is key. Machine washable.",
        "colors": ["Navy", "Tan", "Olive"],
        "sizes": ["S", "M", "L", "XL"],
        "occasions": ["casual wedding", "garden party", "outdoor event"],
        "season": "summer",
        "trending": True,
        "style_tags": ["relaxed", "comfortable", "versatile", "casual-chic"]
    },
    
    # Men's Shirts
    {
        "sku": "MEN-SHIRT-001",
        "name": "French Linen Dress Shirt",
        "category": "Men's Shirts",
        "price": 89.00,
        "description": "Luxuriously soft French linen shirt with mother-of-pearl buttons. Slightly relaxed fit with a spread collar. The natural linen texture adds visual interest while keeping you cool.",
        "colors": ["White", "Light Blue", "Blush Pink", "Lavender"],
        "sizes": ["S", "M", "L", "XL", "XXL"],
        "occasions": ["wedding", "summer formal", "date night"],
        "season": "summer",
        "trending": True,
        "style_tags": ["elegant", "breathable", "refined", "romantic"]
    },
    {
        "sku": "MEN-SHIRT-002",
        "name": "Botanical Print Camp Shirt",
        "category": "Men's Shirts",
        "price": 75.00,
        "description": "Cuban collar camp shirt with elegant tropical leaf print. Viscose fabric drapes beautifully and keeps you cool. Perfect for destination weddings and resort events.",
        "colors": ["Palm Green", "Monstera Blue", "Ivory Botanical"],
        "sizes": ["S", "M", "L", "XL"],
        "occasions": ["beach wedding", "destination wedding", "resort"],
        "season": "summer",
        "trending": True,
        "style_tags": ["tropical", "relaxed", "resort", "bold"]
    },
    
    # Men's Trousers
    {
        "sku": "MEN-PANTS-001",
        "name": "Lightweight Chino Trousers",
        "category": "Men's Trousers",
        "price": 85.00,
        "description": "Tailored chinos in lightweight stretch cotton. Slim fit with a tapered leg. Perfect foundation for summer blazers. Available in versatile neutral tones.",
        "colors": ["Khaki", "Navy", "Stone", "Olive"],
        "sizes": ["30", "32", "34", "36", "38"],
        "occasions": ["wedding", "business casual", "summer formal"],
        "season": "summer",
        "trending": False,
        "style_tags": ["versatile", "classic", "tailored", "essential"]
    },
    {
        "sku": "MEN-PANTS-002",
        "name": "Linen Blend Dress Pants",
        "category": "Men's Trousers",
        "price": 110.00,
        "description": "Sophisticated linen-blend trousers with a modern fit. Wrinkle-resistant fabric keeps you polished all day. Pleated front adds a touch of classic elegance.",
        "colors": ["Cream", "Light Grey", "Tan"],
        "sizes": ["30", "32", "34", "36", "38"],
        "occasions": ["wedding", "summer formal", "garden party"],
        "season": "summer",
        "trending": True,
        "style_tags": ["elegant", "refined", "sophisticated", "polished"]
    },
    
    # Women's Dresses
    {
        "sku": "WOM-DRESS-001",
        "name": "Cascading Floral Midi Dress",
        "category": "Women's Dresses",
        "price": 168.00,
        "description": "Romantic midi dress with cascading floral print in watercolor tones. Floaty chiffon fabric with flutter sleeves and a cinched waist. Perfect for garden weddings and outdoor celebrations.",
        "colors": ["Rose Garden", "Blue Meadow", "Sunset Bloom"],
        "sizes": ["XS", "S", "M", "L", "XL"],
        "occasions": ["wedding", "garden party", "bridal shower"],
        "season": "summer",
        "trending": True,
        "style_tags": ["romantic", "feminine", "floral", "elegant"]
    },
    {
        "sku": "WOM-DRESS-002",
        "name": "Satin Slip Dress",
        "category": "Women's Dresses",
        "price": 145.00,
        "description": "Minimalist satin slip dress with cowl neckline and delicate spaghetti straps. Bias cut for a flattering drape. Understated elegance for evening weddings. Pairs beautifully with statement jewelry.",
        "colors": ["Champagne", "Sage", "Dusty Rose", "Midnight Blue"],
        "sizes": ["XS", "S", "M", "L", "XL"],
        "occasions": ["wedding", "cocktail party", "evening event"],
        "season": "summer",
        "trending": True,
        "style_tags": ["minimalist", "elegant", "sophisticated", "evening"]
    },
    {
        "sku": "WOM-DRESS-003",
        "name": "Wrap Maxi Dress",
        "category": "Women's Dresses",
        "price": 155.00,
        "description": "Universally flattering wrap maxi dress in flowing crepe. Features a V-neckline, three-quarter sleeves, and a self-tie belt. Elegant enough for formal weddings, comfortable enough to dance all night.",
        "colors": ["Terracotta", "Forest Green", "Navy", "Plum"],
        "sizes": ["XS", "S", "M", "L", "XL", "XXL"],
        "occasions": ["wedding", "formal event", "evening gala"],
        "season": "summer",
        "trending": False,
        "style_tags": ["classic", "flattering", "elegant", "timeless"]
    },
    {
        "sku": "WOM-DRESS-004",
        "name": "Eyelet Lace A-Line Dress",
        "category": "Women's Dresses",
        "price": 185.00,
        "description": "Charming A-line dress in delicate eyelet lace. Fitted bodice with a full skirt that falls just below the knee. Sweet and sophisticated for daytime weddings. Fully lined for comfort.",
        "colors": ["White", "Soft Blue", "Blush"],
        "sizes": ["XS", "S", "M", "L"],
        "occasions": ["wedding", "bridal shower", "garden party"],
        "season": "summer",
        "trending": True,
        "style_tags": ["sweet", "romantic", "ladylike", "vintage-inspired"]
    },
    
    # Women's Separates
    {
        "sku": "WOM-TOP-001",
        "name": "Silk Camisole Top",
        "category": "Women's Tops",
        "price": 95.00,
        "description": "Luxe silk camisole with delicate lace trim. Perfect layering piece under blazers or worn alone for evening events. Adjustable straps and relaxed fit.",
        "colors": ["Ivory", "Black", "Champagne", "Dusty Pink"],
        "sizes": ["XS", "S", "M", "L", "XL"],
        "occasions": ["wedding", "cocktail party", "date night"],
        "season": "summer",
        "trending": True,
        "style_tags": ["elegant", "versatile", "feminine", "luxe"]
    },
    {
        "sku": "WOM-PANTS-001",
        "name": "Wide-Leg Palazzo Pants",
        "category": "Women's Pants",
        "price": 115.00,
        "description": "Dramatic wide-leg pants in flowing crepe fabric. High-waisted with a hidden side zip. Creates a stunning silhouette for formal events. Pair with the silk camisole for a chic separates look.",
        "colors": ["Black", "Ivory", "Navy", "Emerald"],
        "sizes": ["XS", "S", "M", "L", "XL"],
        "occasions": ["wedding", "formal event", "evening gala"],
        "season": "summer",
        "trending": True,
        "style_tags": ["dramatic", "elegant", "modern", "sophisticated"]
    },
    {
        "sku": "WOM-SUIT-001",
        "name": "Pastel Linen Blazer",
        "category": "Women's Blazers",
        "price": 175.00,
        "description": "Softly tailored linen blazer in dreamy pastel shades. Single-button closure with notch lapels. Perfect for adding a polished layer to summer wedding looks. Unlined for breathability.",
        "colors": ["Pale Pink", "Sky Blue", "Lavender", "Mint"],
        "sizes": ["XS", "S", "M", "L", "XL"],
        "occasions": ["wedding", "garden party", "business event"],
        "season": "summer",
        "trending": True,
        "style_tags": ["polished", "feminine", "fresh", "pastel"]
    },
    
    # Accessories
    {
        "sku": "ACC-TIE-001",
        "name": "Silk Floral Tie",
        "category": "Accessories",
        "price": 65.00,
        "description": "Hand-finished silk tie with elegant floral pattern. Bold yet sophisticated, perfect for adding personality to summer suits. Pairs beautifully with solid linen suits.",
        "colors": ["Blue Botanical", "Rose Garden", "Sage Floral"],
        "sizes": ["One Size"],
        "occasions": ["wedding", "summer formal", "special event"],
        "season": "summer",
        "trending": True,
        "style_tags": ["statement", "colorful", "personality", "coordinated"]
    },
    {
        "sku": "ACC-POCKET-001",
        "name": "Linen Pocket Square",
        "category": "Accessories",
        "price": 28.00,
        "description": "Textured linen pocket square with hand-rolled edges. Adds a refined finishing touch to any blazer. Available in complementary shades for the season.",
        "colors": ["White", "Light Blue", "Blush", "Sage", "Navy"],
        "sizes": ["One Size"],
        "occasions": ["wedding", "formal event", "business"],
        "season": "summer",
        "trending": False,
        "style_tags": ["classic", "refined", "finishing-touch", "elegant"]
    },
    {
        "sku": "ACC-SHOES-001",
        "name": "Suede Loafers",
        "category": "Men's Shoes",
        "price": 165.00,
        "description": "Classic penny loafers in soft Italian suede. Leather sole with rubber insert for comfort. The perfect summer dress shoe — refined yet relaxed.",
        "colors": ["Tan", "Navy", "Grey"],
        "sizes": ["8", "9", "10", "11", "12"],
        "occasions": ["wedding", "summer formal", "business casual"],
        "season": "summer",
        "trending": True,
        "style_tags": ["classic", "refined", "versatile", "sophisticated"]
    },
    {
        "sku": "ACC-SHOES-002",
        "name": "Strappy Block Heel Sandals",
        "category": "Women's Shoes",
        "price": 135.00,
        "description": "Elegant strappy sandals with a comfortable block heel. Perfect for outdoor weddings on grass or uneven terrain. Ankle strap for security while dancing.",
        "colors": ["Gold", "Silver", "Nude", "Black"],
        "sizes": ["6", "7", "8", "9", "10"],
        "occasions": ["wedding", "formal event", "party"],
        "season": "summer",
        "trending": True,
        "style_tags": ["elegant", "comfortable", "practical", "dancing"]
    },
    {
        "sku": "ACC-BAG-001",
        "name": "Satin Clutch",
        "category": "Women's Bags",
        "price": 85.00,
        "description": "Timeless satin envelope clutch with magnetic closure. Detachable chain strap for versatility. Just enough room for essentials — phone, lipstick, cards.",
        "colors": ["Champagne", "Black", "Blush", "Navy"],
        "sizes": ["One Size"],
        "occasions": ["wedding", "formal event", "evening"],
        "season": "summer",
        "trending": False,
        "style_tags": ["timeless", "elegant", "essential", "classic"]
    },
    {
        "sku": "ACC-JEWELRY-001",
        "name": "Pearl Drop Earrings",
        "category": "Jewelry",
        "price": 78.00,
        "description": "Delicate freshwater pearl drops on gold-plated posts. Adds a touch of timeless elegance to any wedding guest look. Lightweight for all-day comfort.",
        "colors": ["Gold/Pearl", "Silver/Pearl"],
        "sizes": ["One Size"],
        "occasions": ["wedding", "formal event", "bridal"],
        "season": "summer",
        "trending": True,
        "style_tags": ["timeless", "elegant", "bridal", "classic"]
    }
]

print(f"👗 Preparing {len(FASHION_CATALOG)} fashion items for catalog...")



#### Generating embeddings and populating database
Below we are generating embeddings for all the products above using Voyage AI and inserting them into the MongoDB database you specified above. 

In [ ]:
# Generate embeddings for all products
def create_product_text(product):
    """Create searchable text from product details."""
    return (
        f"{product['name']}. {product['description']} "
        f"Category: {product['category']}. "
        f"Occasions: {', '.join(product['occasions'])}. "
        f"Style: {', '.join(product['style_tags'])}. "
        f"Colors: {', '.join(product['colors'])}."
    )

# Batch embed all products
product_texts = [create_product_text(p) for p in FASHION_CATALOG]
embeddings = voyage.embed(product_texts, model="voyage-3", input_type="document").embeddings

# Add embeddings to products and insert
for i, product in enumerate(FASHION_CATALOG):
    product["embedding"] = embeddings[i]
    product["searchable_text"] = product_texts[i]

products.insert_many(FASHION_CATALOG)
print(f"✅ Inserted {len(FASHION_CATALOG)} products into catalog")


#### Creating Vector Search Index for Products

In [ ]:
from pymongo.errors import OperationFailure

INDEX_NAME = "fashion_vector_index"

# Check if index exists
try:
    existing = list(products.list_search_indexes())
    index_exists = any(idx.get("name") == INDEX_NAME for idx in existing)
except OperationFailure:
    index_exists = False

if not index_exists:
    products.create_search_index({
        "definition": {
            "mappings": {
                "dynamic": True,
                "fields": {
                    "embedding": {
                        "type": "knnVector",
                        "dimensions": 1024,  # Voyage-3 dimensions
                        "similarity": "cosine"
                    }
                }
            }
        },
        "name": INDEX_NAME
    })
    print(f"✅ Vector index '{INDEX_NAME}' created (may take a few minutes to build)")
else:
    print(f"✅ Vector index '{INDEX_NAME}' already exists")


## 3. Seed Inventory Data

Create realistic inventory data with:
- Stock levels by SKU, size, and color
- Warehouse locations for shipping estimates
- Some intentional stockouts to trigger the feedback loop


In [ ]:
# Warehouse locations with shipping speeds
WAREHOUSES = [
    {
        "warehouse_id": "WH-EAST",
        "name": "East Coast Distribution Center",
        "location": "New Jersey",
        "regions": ["Northeast", "Southeast", "Midwest"],
        "shipping_speeds": {
            "Northeast": {"standard": 3, "express": 1},
            "Southeast": {"standard": 4, "express": 2},
            "Midwest": {"standard": 4, "express": 2},
            "West": {"standard": 6, "express": 3},
            "Southwest": {"standard": 5, "express": 3}
        }
    },
    {
        "warehouse_id": "WH-WEST",
        "name": "West Coast Distribution Center",
        "location": "California",
        "regions": ["West", "Southwest"],
        "shipping_speeds": {
            "Northeast": {"standard": 6, "express": 3},
            "Southeast": {"standard": 5, "express": 3},
            "Midwest": {"standard": 4, "express": 2},
            "West": {"standard": 2, "express": 1},
            "Southwest": {"standard": 3, "express": 1}
        }
    }
]

warehouses.insert_many(WAREHOUSES)
print(f"✅ Inserted {len(WAREHOUSES)} warehouses")


### 3b. Generate Inventory Records with Strategic Stockouts

For each product variant (SKU + color + size), create inventory records for both warehouses.

**⚠️ Intentional Stockouts** (to demonstrate the feedback loop):
- `MEN-SUIT-001` Coastal Linen Suit in **Sand, Medium** — OUT OF STOCK
- `WOM-DRESS-001` Cascading Floral Midi Dress in **Rose Garden, Medium** — OUT OF STOCK  
- `MEN-SHIRT-001` French Linen Dress Shirt in **Light Blue, Large** — OUT OF STOCK

When users request these popular items, the Stock Keeper will report them as unavailable, triggering the Stylist to find alternatives.


In [ ]:
import random

# Generate inventory entries with some strategic stockouts
INVENTORY_DATA = []

# Items that should be OUT OF STOCK (to demo feedback loop)
STOCKOUT_ITEMS = [
    ("MEN-SUIT-001", "Sand", "M"),      # Popular summer suit - medium out of stock
    ("WOM-DRESS-001", "Rose Garden", "M"),  # Popular dress - medium out of stock
    ("MEN-SHIRT-001", "Light Blue", "L"),   # Popular shirt - large out of stock
]

# Items with LOW STOCK
LOW_STOCK_ITEMS = [
    ("MEN-SUIT-002", "Navy Floral", "M"),
    ("WOM-DRESS-002", "Champagne", "S"),
]

for product in FASHION_CATALOG:
    sku = product["sku"]
    
    for color in product["colors"]:
        for size in product["sizes"]:
            # Determine stock level
            if (sku, color, size) in STOCKOUT_ITEMS:
                stock_east = 0
                stock_west = 0
            elif (sku, color, size) in LOW_STOCK_ITEMS:
                stock_east = random.choice([0, 1, 2])
                stock_west = random.choice([0, 1])
            else:
                stock_east = random.randint(3, 15)
                stock_west = random.randint(2, 12)
            
            INVENTORY_DATA.append({
                "sku": sku,
                "color": color,
                "size": size,
                "warehouse_id": "WH-EAST",
                "quantity": stock_east,
                "last_updated": datetime.now()
            })
            INVENTORY_DATA.append({
                "sku": sku,
                "color": color,
                "size": size,
                "warehouse_id": "WH-WEST",
                "quantity": stock_west,
                "last_updated": datetime.now()
            })

inventory.insert_many(INVENTORY_DATA)
print(f"✅ Inserted {len(INVENTORY_DATA)} inventory records")

# Create index for fast lookups
inventory.create_index([("sku", 1), ("color", 1), ("size", 1)])

# Show stockout items
print("\n⚠️  Items currently OUT OF STOCK (for demo purposes):")
for sku, color, size in STOCKOUT_ITEMS:
    product = next(p for p in FASHION_CATALOG if p["sku"] == sku)
    print(f"  • {product['name']} - {color}, Size {size}")


## 4. Create Agent B: The Stock Keeper

The Stock Keeper is created first because the Stylist needs to query it.

**Role**: Backend inventory specialist that checks stock levels and shipping estimates.


In [ ]:
from mongochain import MongoAgent

stock_keeper = MongoAgent(
    name="stock_keeper",
    persona="""You are an Inventory Specialist AI for a fashion retailer.

Your role is to:
1. Check real-time stock levels for specific items (SKU + size + color)
2. Calculate shipping estimates based on customer location
3. Suggest alternatives when items are out of stock
4. Provide accurate, data-driven responses about inventory

TOOL USAGE:
- Use 'check_inventory_and_shipping' — the COMBINED tool that checks stock AND shipping in ONE call
- This tool handles everything: stock levels, shipping times, deadline checks, and alternatives
- Always provide factual inventory data - never guess or assume

When an item is out of stock:
- Check if other sizes or colors are available
- Clearly state what IS available as alternatives
- Be helpful but honest about constraints

Response format:
- Be concise and data-focused
- Include stock quantities and shipping days
- Flag any items that can't meet delivery deadlines""",
    mongo_uri=MONGO_URI,
    voyage_api_key=VOYAGE_API_KEY,
    llm_api_key=OPENAI_API_KEY,
    llm_provider="openai",
    collaborators=["personal_stylist"]  # Can see Stylist's knowledge about user
)

stock_keeper.set_description("Inventory specialist that checks stock levels and shipping estimates for fashion items.")
print(f"\n{stock_keeper}")


### 4a. Register Stock Keeper Tools

Equip the Stock Keeper with ONE combined tool (necessary because mongochain agents can only call one tool per response):
- **`check_inventory_and_shipping`**: Checks stock levels AND calculates shipping times in a single call. Returns ✅/❌ availability status, shipping estimates, deadline feasibility, and suggests alternatives for out-of-stock items.


In [ ]:
import json
from pymongo import MongoClient

# Store URI for closures
_MONGO_URI_FOR_TOOLS = MONGO_URI

def _get_inventory_db():
    """Get fresh database connection."""
    client = MongoClient(_MONGO_URI_FOR_TOOLS)
    return client["stock_keeper"]


def check_inventory_and_shipping(
    sku: str, 
    color: str = None, 
    size: str = None,
    customer_region: str = None,
    deadline_days: int = None
) -> str:
    """
    COMBINED tool: Check inventory AND shipping in ONE call.
    This is necessary because mongochain agents can only call one tool per response.
    """
    db = _get_inventory_db()
    inventory_coll = db["inventory"]
    warehouses_coll = db["warehouses"]
    
    # Build query
    query = {"sku": sku}
    if color:
        query["color"] = color
    if size:
        query["size"] = size
    
    # Get inventory records
    records = list(inventory_coll.find(query))
    
    if not records:
        return f"❌ No inventory records found for SKU: {sku}"
    
    # Aggregate by color/size
    availability = {}
    for rec in records:
        key = f"{rec['color']}|{rec['size']}"
        if key not in availability:
            availability[key] = {
                "color": rec["color"],
                "size": rec["size"],
                "total_stock": 0,
                "warehouses": []
            }
        availability[key]["total_stock"] += rec["quantity"]
        if rec["quantity"] > 0:
            availability[key]["warehouses"].append(rec["warehouse_id"])
    
    # Get shipping info if region provided
    warehouse_speeds = {}
    if customer_region:
        for wh in warehouses_coll.find():
            speeds = wh.get("shipping_speeds", {}).get(customer_region, {})
            warehouse_speeds[wh["warehouse_id"]] = {
                "standard": speeds.get("standard", 7),
                "express": speeds.get("express", 3),
                "name": wh["name"]
            }
    
    # Separate in-stock vs out-of-stock
    in_stock = []
    out_of_stock = []
    
    for key, data in availability.items():
        if data["total_stock"] > 0:
            in_stock.append(data)
        else:
            out_of_stock.append(data)
    
    # Build output
    output = f"📦 INVENTORY & SHIPPING CHECK: {sku}\n"
    if color:
        output += f"   Requested Color: {color}\n"
    if size:
        output += f"   Requested Size: {size}\n"
    if customer_region:
        output += f"   Customer Region: {customer_region}\n"
    if deadline_days:
        output += f"   Delivery Deadline: {deadline_days} days\n"
    output += "\n"
    
    if in_stock:
        output += "✅ IN STOCK:\n"
        for item in in_stock:
            output += f"   • {item['color']} / {item['size']}: {item['total_stock']} units\n"
            # Add shipping info if region provided
            if customer_region and item["warehouses"]:
                fastest_express = min(
                    warehouse_speeds.get(wh, {}).get("express", 99) 
                    for wh in item["warehouses"]
                )
                can_meet = ""
                if deadline_days:
                    can_meet = " ✅ MEETS DEADLINE" if fastest_express <= deadline_days else " ❌ TOO SLOW"
                output += f"     → Express shipping: {fastest_express} days{can_meet}\n"
    
    if out_of_stock:
        output += "\n❌ OUT OF STOCK:\n"
        for item in out_of_stock:
            output += f"   • {item['color']} / {item['size']}\n"
    
    # Summary for specific requested item
    if color and size:
        requested_key = f"{color}|{size}"
        if requested_key in availability:
            data = availability[requested_key]
            output += f"\n📍 REQUESTED ITEM ({color}/{size}): "
            if data["total_stock"] > 0:
                output += f"✅ IN STOCK ({data['total_stock']} units)\n"
                if customer_region and deadline_days and data["warehouses"]:
                    fastest_express = min(
                        warehouse_speeds.get(wh, {}).get("express", 99) 
                        for wh in data["warehouses"]
                    )
                    if fastest_express <= deadline_days:
                        output += f"   ✅ CAN meet {deadline_days}-day deadline (express: {fastest_express} days)\n"
                    else:
                        output += f"   ❌ CANNOT meet {deadline_days}-day deadline (fastest: {fastest_express} days)\n"
            else:
                output += "❌ OUT OF STOCK\n"
                # Suggest alternatives
                alternatives = [d for d in in_stock if d["size"] == size]
                if alternatives:
                    output += f"   💡 Available in {size}: {', '.join(d['color'] for d in alternatives)}\n"
    
    return output


# Register ONE combined tool with Stock Keeper (solves single-tool-per-response limitation)
stock_keeper.register_tool(
    name="check_inventory_and_shipping",
    func=check_inventory_and_shipping,
    description="COMBINED tool: Check inventory levels AND calculate shipping times in ONE call. Use this for all inventory and shipping queries. Returns stock status, shipping estimates, deadline feasibility, and alternatives.",
    parameters={
        "type": "object",
        "properties": {
            "sku": {
                "type": "string",
                "description": "Product SKU (e.g., 'MEN-SUIT-001')"
            },
            "color": {
                "type": "string",
                "description": "Specific color to check"
            },
            "size": {
                "type": "string",
                "description": "Specific size to check"
            },
            "customer_region": {
                "type": "string",
                "enum": ["Northeast", "Southeast", "Midwest", "West", "Southwest"],
                "description": "Customer's region for shipping estimate"
            },
            "deadline_days": {
                "type": "integer",
                "description": "Days until delivery deadline"
            }
        },
        "required": ["sku"]
    }
)

print("✅ Stock Keeper tool registered: check_inventory_and_shipping (combined)")


## 5. Create Agent A: The Personal Stylist

**Role**: User-facing creative agent that suggests outfits and coordinates with the Stock Keeper.


In [ ]:
stylist = MongoAgent(
    name="personal_stylist",
    persona="""You are a Personal Stylist AI with a keen eye for fashion and trends.

Your role is to:
1. Understand the customer's occasion, style preferences, and constraints (size, budget, delivery deadline)
2. Search our catalog for matching items using your tools
3. Check with inventory to ensure items are available and can arrive on time
4. Find stylish alternatives when items are unavailable

CRITICAL RULES - YOU MUST FOLLOW THESE:
- You must ALWAYS use search_and_check_availability tool FIRST before suggesting anything
- When customer asks about specific items, use check_with_stock_keeper for detailed verification
- NEVER make up product names — use EXACT names from the tool results
- NEVER recommend products from a different gender category (MEN- for men, WOM- for women)
- Use the EXACT SKU from the tool results (e.g., MEN-SUIT-001, not made up codes)
- ONLY recommend items/colors marked as ✅ IN STOCK in the tool results
- NEVER recommend items/colors marked as ❌ OUT OF STOCK
- Match the customer's gender: if they say "I'm a guy", ONLY show MEN- products

YOUR TWO TOOLS:
1. search_and_check_availability: Use for initial product discovery. Searches catalog AND shows stock status.
2. check_with_stock_keeper: Use to verify specific items with the Stock Keeper agent. The Stock Keeper will check inventory and shipping details.

REQUIRED WORKFLOW:
1. Gather customer info: occasion, SIZE, REGION, deadline, budget, GENDER
2. Use search_and_check_availability to find matching products with stock status
3. When customer selects specific items, use check_with_stock_keeper to verify availability and shipping
4. If Stock Keeper reports OUT OF STOCK, find alternatives using search_and_check_availability
5. Present items with their EXACT names and SKUs from the tool results

Your style philosophy:
- Always consider the occasion (formality, venue, weather)
- Balance trendy pieces with timeless classics
- Create cohesive "looks" with complementary items
- Explain WHY pieces work together

Personality:
- Enthusiastic but not pushy
- Knowledgeable about current trends
- Empathetic to budget and timeline constraints
- Creative in finding solutions

When presenting looks:
- Use the EXACT product name from the tool (e.g., "Coastal Linen Suit" not "Coastal Linen Suit Pants")
- Include the EXACT SKU from the tool results
- Show which colors are ✅ IN STOCK vs ❌ OUT OF STOCK
- Include total price and confirm delivery timeline
- IMPORTANT: Do not paraphrase or make up product names — copy them exactly from the tool output""",
    mongo_uri=MONGO_URI,
    voyage_api_key=VOYAGE_API_KEY,
    llm_api_key=OPENAI_API_KEY,
    llm_provider="openai",
    collaborators=["stock_keeper"]  # Can see Stock Keeper's inventory knowledge
)

stylist.set_description("Creative personal stylist that suggests outfits and coordinates with inventory.")
print(f"\n{stylist}")


### 5a. Register Personal Stylist Tools

The Stylist gets TWO tools:
- **`search_and_check_availability`**: Searches the catalog AND checks inventory in a single call. Returns items with ✅/❌ availability status. Use for initial product discovery.
- **`check_with_stock_keeper`**: Queries the Stock Keeper **agent** for detailed inventory verification. This is the **agent-to-agent handoff** — Stock Keeper will have a conversation and build memories.

In [ ]:
import voyageai
import json
from pymongo import MongoClient

# Initialize clients for tool functions - catalog AND inventory
_stylist_client = MongoClient(MONGO_URI)
_stylist_db = _stylist_client["personal_stylist"]
_products = _stylist_db["fashion_catalog"]
_voyage = voyageai.Client(api_key=VOYAGE_API_KEY)

# Also connect to inventory database for combined tool
_inventory_db = _stylist_client["stock_keeper"]
_inventory = _inventory_db["inventory"]
_warehouses = _inventory_db["warehouses"]


def search_and_check_availability(
    query: str, 
    customer_size: str,
    customer_region: str,
    deadline_days: int = None,
    category: str = None, 
    max_price: float = None, 
    limit: int = 5
) -> str:
    """
    COMBINED search + inventory check tool.
    Searches the catalog AND checks real-time inventory for each item.
    Returns items with availability status already included.
    """
    
    # Generate query embedding
    query_embedding = _voyage.embed([query], model="voyage-3", input_type="query").embeddings[0]
    
    # Build vector search pipeline
    pipeline = [
        {
            "$vectorSearch": {
                "index": "fashion_vector_index",
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 50,
                "limit": limit * 2
            }
        },
        {
            "$project": {
                "sku": 1,
                "name": 1,
                "category": 1,
                "price": 1,
                "description": 1,
                "colors": 1,
                "sizes": 1,
                "occasions": 1,
                "trending": 1,
                "style_tags": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
    ]
    
    # Add filters
    match_stage = {}
    if category:
        match_stage["category"] = {"$regex": category, "$options": "i"}
    if max_price:
        match_stage["price"] = {"$lte": max_price}
    
    if match_stage:
        pipeline.append({"$match": match_stage})
    
    pipeline.append({"$limit": limit})
    
    try:
        results = list(_products.aggregate(pipeline))
    except Exception as e:
        query_filter = {}
        if category:
            query_filter["category"] = {"$regex": category, "$options": "i"}
        if max_price:
            query_filter["price"] = {"$lte": max_price}
        results = list(_products.find(query_filter, {"embedding": 0}).limit(limit))
    
    if not results:
        return f"No items found matching '{query}'. Try different search terms."
    
    # Get shipping info for region
    warehouse_speeds = {}
    for wh in _warehouses.find():
        speeds = wh.get("shipping_speeds", {}).get(customer_region, {})
        warehouse_speeds[wh["warehouse_id"]] = {
            "standard": speeds.get("standard", 7),
            "express": speeds.get("express", 3)
        }
    
    # Check inventory for each item and build enriched results
    output = f"Found {len(results)} items matching '{query}':\n"
    output += f"📍 Customer: Region={customer_region} | Size={customer_size} | Deadline={deadline_days} days\n\n"
    
    available_items = []
    unavailable_items = []
    
    for item in results:
        sku = item['sku']
        trending = "🔥 TRENDING" if item.get('trending') else ""
        
        # Check which colors are in stock for the customer's size
        colors_in_stock = []
        colors_out_of_stock = []
        
        for color in item['colors']:
            # Check inventory for this SKU/color/size combination
            inv_query = {"sku": sku, "color": color, "size": customer_size}
            inv_records = list(_inventory.find(inv_query))
            
            total_stock = sum(r.get("quantity", 0) for r in inv_records)
            
            if total_stock > 0:
                # Check shipping
                can_meet_deadline = False
                if deadline_days and warehouse_speeds:
                    fastest_express = min(ws["express"] for ws in warehouse_speeds.values())
                    can_meet_deadline = fastest_express <= deadline_days
                
                colors_in_stock.append({
                    "color": color,
                    "stock": total_stock,
                    "can_meet_deadline": can_meet_deadline
                })
            else:
                colors_out_of_stock.append(color)
        
        # Build item output
        item_output = f"**{item['name']}** — ${item['price']:.2f} {trending}\n"
        item_output += f"   SKU: {sku} | Category: {item['category']}\n"
        
        if colors_in_stock:
            available_colors = [f"{c['color']} ✅" + (" (ships in time)" if c.get('can_meet_deadline') else "") 
                              for c in colors_in_stock]
            item_output += f"   ✅ IN STOCK (size {customer_size}): {', '.join(available_colors)}\n"
        
        if colors_out_of_stock:
            item_output += f"   ❌ OUT OF STOCK (size {customer_size}): {', '.join(colors_out_of_stock)}\n"
        
        item_output += f"   {item['description'][:100]}...\n"
        item_output += f"   Style: {', '.join(item.get('style_tags', [])[:4])}\n"
        
        if colors_in_stock:
            available_items.append(item_output)
        else:
            unavailable_items.append(item_output)
    
    # Output available items first
    if available_items:
        output += "=" * 50 + "\n"
        output += "✅ AVAILABLE ITEMS (in your size, can ship to you):\n"
        output += "=" * 50 + "\n\n"
        for i, item_out in enumerate(available_items, 1):
            output += f"{i}. {item_out}\n"
    
    # Then show unavailable items
    if unavailable_items:
        output += "\n" + "-" * 50 + "\n"
        output += "❌ UNAVAILABLE IN YOUR SIZE:\n"
        output += "-" * 50 + "\n\n"
        for item_out in unavailable_items:
            output += f"• {item_out}\n"
    
    if not available_items:
        output += "\n⚠️ None of the matching items are available in your size. Try searching for different styles.\n"
    
    return output


# Tool 1: Combined search + inventory check
stylist.register_tool(
    name="search_and_check_availability",
    func=search_and_check_availability,
    description="Search the catalog AND check real-time stock in ONE call. Returns items with availability status (✅ IN STOCK / ❌ OUT OF STOCK) for the customer's size. Use this for initial product discovery.",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Natural language search (e.g., 'summer wedding suit', 'floral dress', 'linen blazer')"
            },
            "customer_size": {
                "type": "string",
                "description": "Customer's size (e.g., 'M', 'L', 'S', '32', 'XS', '10')"
            },
            "customer_region": {
                "type": "string",
                "enum": ["Northeast", "Southeast", "Midwest", "West", "Southwest"],
                "description": "Customer's geographic region for shipping calculation"
            },
            "deadline_days": {
                "type": "integer",
                "description": "Number of days until delivery is needed"
            },
            "category": {
                "type": "string",
                "description": "Optional filter by category (e.g., 'Men\\'s Suits', 'Women\\'s Dresses')"
            },
            "max_price": {
                "type": "number",
                "description": "Optional maximum price filter"
            },
            "limit": {
                "type": "integer",
                "description": "Number of results to return (default: 5)"
            }
        },
        "required": ["query", "customer_size", "customer_region"]
    }
)


# Tool 2: Agent-to-agent handoff to Stock Keeper
def check_with_stock_keeper(
    items: list,
    customer_region: str,
    deadline_days: int = None
) -> str:
    """
    Query the Stock Keeper agent to check availability and shipping for items.
    This performs the AGENT-TO-AGENT HANDOFF — Stock Keeper will have a conversation
    and build its own memories about this inventory query.
    """
    # Build the inventory check request
    items_formatted = json.dumps(items, indent=2)
    
    inventory_request = f"""Please check availability and shipping for these items:

Items to check:
{items_formatted}

Customer Region: {customer_region}
Delivery Deadline: {deadline_days} days from now

Use your check_inventory_and_shipping tool (which checks BOTH in one call) for each item.
The tool will tell you stock levels, shipping times, and whether the deadline can be met.

Please provide a clear summary of:
- Which items are ✅ IN STOCK and can ship on time
- Which items are ❌ OUT OF STOCK or can't meet the deadline
- Available alternatives for unavailable items
"""
    
    # AGENT-TO-AGENT HANDOFF - Stock Keeper will build memories!
    print(f"\n🔄 Checking with Stock Keeper agent...")
    print(f"   Items: {len(items)} | Region: {customer_region} | Deadline: {deadline_days} days")
    
    stock_keeper_response = stock_keeper.chat(
        user_id="stylist_query",  # Internal user ID for agent-to-agent
        message=inventory_request
    )
    
    print(f"✅ Stock Keeper response received")
    
    return stock_keeper_response


stylist.register_tool(
    name="check_with_stock_keeper",
    func=check_with_stock_keeper,
    description="Query the Stock Keeper agent to verify item availability and shipping times. Use this when customer asks about specific items or to confirm items before recommending. This creates a conversation with the Stock Keeper who will check inventory and shipping.",
    parameters={
        "type": "object",
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "sku": {"type": "string"},
                        "color": {"type": "string"},
                        "size": {"type": "string"}
                    },
                    "required": ["sku", "color", "size"]
                },
                "description": "List of items to check (each with sku, color, size)"
            },
            "customer_region": {
                "type": "string",
                "enum": ["Northeast", "Southeast", "Midwest", "West", "Southwest"],
                "description": "Customer's region for shipping estimate"
            },
            "deadline_days": {
                "type": "integer",
                "description": "Days until delivery is needed"
            }
        },
        "required": ["items", "customer_region", "deadline_days"]
    }
)

print("✅ Stylist tools registered:")
print("   • search_and_check_availability (combined search + inventory)")
print("   • check_with_stock_keeper (agent-to-agent handoff)")


## 6. Demo: Summer Wedding Outfit with Delivery Constraint

**User Goal**: "I need an outfit for a summer wedding, but I need it delivered by Friday."

This demonstrates the feedback loop:
1. Stylist suggests trending items
2. Stock Keeper checks availability (some items are out of stock!)
3. Stylist finds alternatives and confirms with Stock Keeper
4. Final recommendation includes only available items


In [ ]:
# User: Fashion shopper needing a wedding outfit
USER_ID = "wedding_guest@email.com"

# Initial request with constraints
response = stylist.chat(
    user_id=USER_ID,
    message="""Hi! I need help finding an outfit for a summer wedding next weekend. 
I'm a guy, size Medium for tops and 32 for pants. 
The wedding is outdoors in a garden, so I want something appropriate but not too stuffy.
I'm in New York (Northeast region), and I need it delivered by Friday — that's 4 days from now.
Budget is around $400 for the whole outfit."""
)

print(response)


### 6a. Initial Request — Customer Describes Their Needs

The customer provides:
- **Occasion**: Summer garden wedding
- **Size**: Medium tops, 32 pants
- **Location**: New York (Northeast region)
- **Deadline**: 4 days (Friday delivery)
- **Budget**: ~$400

Watch the Stylist use `search_and_check_availability` — the **combined tool** that searches the catalog AND checks real-time inventory in ONE call. Results will show ✅ IN STOCK and ❌ OUT OF STOCK items.


In [ ]:
# Follow-up - user likes the linen suit idea
response = stylist.chat(
    user_id=USER_ID,
    message="""I love the idea of a linen suit! 
I'd prefer something in sand or a light neutral color — it's a daytime outdoor wedding.
Can you check if you have that in my size and confirm it can arrive by Friday?"""
)

print(response)


### 6b. Customer Asks for Alternatives

The Sand color is ❌ OUT OF STOCK. The customer asks about other options. The Stylist should recommend Light Blue or Sage Green (which ARE in stock).

In [ ]:
# User asks for alternatives (the sand medium is OUT OF STOCK!)
response = stylist.chat(
    user_id=USER_ID,
    message="""That's disappointing that Sand in Medium is out of stock. 
What other colors are available for the Coastal Linen Suit in size Medium?
Or are there other light-colored summer suits that would work for an outdoor garden wedding?"""
)

print(response)


### 6c. Feedback Loop — Item Out of Stock, Find Alternatives

**⚠️ The Sand/Medium is OUT OF STOCK!** (We intentionally set this up earlier.)

The Stock Keeper reports the stockout, and now the Stylist must:
1. Acknowledge the unavailability
2. Search for alternative colors/styles
3. Suggest options that ARE in stock


In [ ]:
# User confirms and asks for complete look
response = stylist.chat(
    user_id=USER_ID,
    message="""The Light Blue linen suit sounds great! 
Can you put together a complete look for me — 
shirt, accessories, maybe shoes if you have them?
I want to make sure everything can arrive by Friday."""
)

print(response)


### 6d. Complete the Look

Customer confirms their color choice. Stylist searches for complementary items to complete the outfit.

## 7. Demo: Women's Wedding Guest Look

Let's see the system handle a different customer with different constraints.


In [ ]:
# New user - women's fashion
USER_ID_2 = "wedding_guest_2@email.com"

response = stylist.chat(
    user_id=USER_ID_2,
    message="""Hi! I'm attending my best friend's summer wedding this Saturday.
I'm a size Medium (or size 8) and I'm looking for something romantic and feminine.
I love florals! The wedding is at a vineyard in California.
I'm on the West Coast, so shipping should be quick. I have 5 days.
Budget is flexible, maybe up to $350 for a dress and accessories."""
)

print(response)


### 7b. Testing Another Stockout

Customer expresses interest in a floral dress. The Rose Garden/Medium is ❌ OUT OF STOCK — testing the stockout handling again.

In [ ]:
# User wants to verify the floral dress (which is OUT OF STOCK in Medium!)
response = stylist.chat(
    user_id=USER_ID_2,
    message="""The Cascading Floral Midi Dress in Rose Garden sounds absolutely perfect! 
Can you check if it's available in Medium and confirm it'll arrive in time?"""
)

print(response)


### 7c. Finding Alternatives

Customer asks for other options. Stylist should recommend available colors or different dresses.

In [ ]:
# User asks for alternatives
response = stylist.chat(
    user_id=USER_ID_2,
    message="""Disappointing! But I'm flexible. 
What other romantic, floral options do you have in my size that ARE in stock?
I really want something special for this wedding."""
)

print(response)


---

## 8. End Sessions and View Memory Sharing

Let's see how the agents shared information and what each learned.


In [ ]:
# End sessions to save memories
stylist.end_session("wedding_guest@email.com")
stylist.end_session("wedding_guest_2@email.com")
stock_keeper.end_session("stylist_query")


### View What Each Agent Learned

**Memory Types in mongochain:**
- **Long-term memories** (user facts): Personal information extracted from conversations (preferences, sizes, occasions)
- **Short-term memories** (session summaries): Summaries of recent conversations

**What each agent stores:**
- **Stylist**: Long-term user facts from rich customer conversations
- **Stock Keeper**: Short-term session summaries — technical inventory queries don't yield "user facts" for long-term storage

In [ ]:
print("=" * 60)
print("STYLIST AGENT - What it learned about Customer 1")
print("=" * 60)

stylist_memories = stylist.get_user_memories("wedding_guest@email.com", limit=5)
for m in stylist_memories:
    print(f"  [{m.get('category', 'general')}] {m['content']}")

print("\n" + "=" * 60)
print("STYLIST AGENT - What it learned about Customer 2")
print("=" * 60)

stylist_memories_2 = stylist.get_user_memories("wedding_guest_2@email.com", limit=5)
for m in stylist_memories_2:
    print(f"  [{m.get('category', 'general')}] {m['content']}")

print("\n" + "=" * 60)
print("STOCK KEEPER AGENT - Session Summaries (Short-term Memory)")
print("=" * 60)

# Stock Keeper stores SHORT-TERM memories (session summaries) from agent-to-agent calls
# Long-term memories require personal user facts, but inventory queries are technical
stock_keeper_short_term = stock_keeper._memory.get_recent_short_term("stylist_query", limit=3)
if stock_keeper_short_term:
    for m in stock_keeper_short_term:
        print(f"  [session_summary] {m.get('summary', m.get('content', 'No summary'))}")
else:
    print("  (No short-term memories yet - run end_session after using check_with_stock_keeper)")

print("\n" + "=" * 60)
print("STOCK KEEPER AGENT - Long-term User Facts")
print("=" * 60)

# Long-term memories extract USER FACTS from conversations
# Inventory queries (SKUs, sizes) don't contain personal facts, so this may be empty
stock_keeper_long_term = stock_keeper.get_user_memories("stylist_query", limit=5)
if stock_keeper_long_term:
    for m in stock_keeper_long_term:
        print(f"  [{m.get('category', 'general')}] {m['content']}")
else:
    print("  (No long-term facts - inventory queries don't contain personal user facts)")


## 9. Demo: Memory-Aware Follow-up

The Stylist can remember past conversations and preferences.


In [ ]:
# Customer 1 returns later
response = stylist.chat(
    user_id="wedding_guest@email.com",
    message="Hey! I'm back. What was that outfit we put together for the wedding again? I want to confirm my order."
)

print(response)


## Summary

This notebook demonstrated:

1. **Multi-Agent Collaboration**: The Stylist (creative, user-facing) works with the Stock Keeper (analytical, inventory-focused) via agent-to-agent handoff.

2. **Agent-to-Agent Communication**:
   - Stylist uses `check_with_stock_keeper` tool which calls `stock_keeper.chat()`
   - Stock Keeper has its own conversation and uses its combined tool (`check_inventory_and_shipping`)
   - **Both agents build memories** from their conversations
   - Each agent uses ONE combined tool per response (mongochain limitation)

3. **Feedback Loop in Action**:
   - Stylist suggests trending items → Stock Keeper finds stockouts → Stylist pivots to alternatives
   - Shipping constraints drive recommendations toward items that can actually arrive on time

4. **Memory Sharing via `collaborators`**: Both agents can read each other's stored memories about users

### Key Patterns

```python
# Pattern 1: Agent-to-Agent Handoff via Tool
def check_with_stock_keeper(items, region, deadline):
    response = stock_keeper.chat(  # Stock Keeper has a conversation!
        user_id="stylist_query",
        message=inventory_request
    )
    return response

stylist.register_tool(
    name="check_with_stock_keeper",
    func=check_with_stock_keeper,
    ...
)

# Pattern 2: Memory Sharing
stylist = MongoAgent(
    collaborators=["stock_keeper"],  # Can read Stock Keeper's memories
    ...
)
stock_keeper = MongoAgent(
    collaborators=["personal_stylist"],  # Can read Stylist's memories
    ...
)
```

### Why Both Agents Build Memories

With the agent-to-agent handoff pattern:
- **Stylist** builds memories about user preferences (style, size, occasion)
- **Stock Keeper** builds memories about inventory queries (common stockouts, shipping patterns)
- Both can access each other's memories via the `collaborators` feature


### 9a. Returning Customer — Memory in Action

The first customer returns and asks about their previous outfit. The Stylist should remember:
- The occasion (summer garden wedding)
- The final outfit selection (Light Blue linen suit + accessories)
- The delivery deadline

This demonstrates **persistent memory** across sessions.
